In [1]:
# ============================================================
# TASK 1 – Comparative LLM Fine-Tuning Project
# Dataset: GoEmotions (mrm8488/goemotions)
# Baseline Model: google/flan-t5-small
# ============================================================

# Install required libraries
!pip install -q transformers datasets accelerate evaluate sentencepiece bitsandbytes

# Load dataset
from datasets import load_dataset

ds = load_dataset("mrm8488/goemotions")
ds

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 10.8 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


goemotions.csv:   0%|          | 0.00/42.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/211225 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'id', 'author', 'subreddit', 'link_id', 'parent_id', 'created_utc', 'rater_id', 'example_very_unclear', 'admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval', 'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief', 'joy', 'love', 'nervousness', 'optimism', 'pride', 'realization', 'relief', 'remorse', 'sadness', 'surprise', 'neutral'],
        num_rows: 211225
    })
})

In [2]:
# Inspect sample
print(ds["train"][0])

{'text': 'That game hurt.', 'id': 'eew5j0j', 'author': 'Brdd9', 'subreddit': 'nrl', 'link_id': 't3_ajis4z', 'parent_id': 't1_eew18eq', 'created_utc': 1548381039.0, 'rater_id': 1, 'example_very_unclear': False, 'admiration': 0, 'amusement': 0, 'anger': 0, 'annoyance': 0, 'approval': 0, 'caring': 0, 'confusion': 0, 'curiosity': 0, 'desire': 0, 'disappointment': 0, 'disapproval': 0, 'disgust': 0, 'embarrassment': 0, 'excitement': 0, 'fear': 0, 'gratitude': 0, 'grief': 0, 'joy': 0, 'love': 0, 'nervousness': 0, 'optimism': 0, 'pride': 0, 'realization': 0, 'relief': 0, 'remorse': 0, 'sadness': 1, 'surprise': 0, 'neutral': 0}


In [9]:
# Load emotion map
feature_names = list(ds["train"].features.keys())
start_index = feature_names.index('admiration')
end_index = feature_names.index('neutral')
emotion_labels = feature_names[start_index : end_index+1]
emotion_labels

['admiration',
 'amusement',
 'anger',
 'annoyance',
 'approval',
 'caring',
 'confusion',
 'curiosity',
 'desire',
 'disappointment',
 'disapproval',
 'disgust',
 'embarrassment',
 'excitement',
 'fear',
 'gratitude',
 'grief',
 'joy',
 'love',
 'nervousness',
 'optimism',
 'pride',
 'realization',
 'relief',
 'remorse',
 'sadness',
 'surprise',
 'neutral']

In [11]:
# Convert labels list → comma-separated string
def format_example(example):
    active_labels = []
    for emotion in emotion_labels:
        if example[emotion] == 1:
            active_labels.append(emotion)
    example["label_text"] = ", ".join(active_labels)
    return example

ds = ds.map(format_example)
ds

Map:   0%|          | 0/211225 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'id', 'author', 'subreddit', 'link_id', 'parent_id', 'created_utc', 'rater_id', 'example_very_unclear', 'admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval', 'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief', 'joy', 'love', 'nervousness', 'optimism', 'pride', 'realization', 'relief', 'remorse', 'sadness', 'surprise', 'neutral', 'label_text'],
        num_rows: 211225
    })
})

In [13]:
# GoEmotions dataset only has a 'train' split. We will create train, validation, and test splits from it.

# Step 1: Split the original 'train' into a combined train/validation set and a separate test set.
# This results in roughly 90% for train/val and 10% for test of the original dataset.
train_val_split_result = ds["train"].train_test_split(test_size=0.1, seed=42)
train_val_set = train_val_split_result["train"]
test_set = train_val_split_result["test"]

# Step 2: Split the 'train_val_set' further into the final train set and validation set.
# To achieve an approximate 80% train, 10% validation, 10% test split of the original dataset,
# we need the validation set to be 10% of the *original* dataset.
# Since train_val_set is 90% of the original, 10% of original is (0.1 / 0.9) = 1/9 of train_val_set.
train_val_split_final = train_val_set.train_test_split(test_size=1/9, seed=42)
train_set = train_val_split_final["train"]
val_set = train_val_split_final["test"]

train_set, val_set, test_set

(Dataset({
     features: ['text', 'id', 'author', 'subreddit', 'link_id', 'parent_id', 'created_utc', 'rater_id', 'example_very_unclear', 'admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval', 'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief', 'joy', 'love', 'nervousness', 'optimism', 'pride', 'realization', 'relief', 'remorse', 'sadness', 'surprise', 'neutral', 'label_text'],
     num_rows: 168979
 }),
 Dataset({
     features: ['text', 'id', 'author', 'subreddit', 'link_id', 'parent_id', 'created_utc', 'rater_id', 'example_very_unclear', 'admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval', 'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief', 'joy', 'love', 'nervousness', 'optimism', 'pride', 'realization', 'relief', 'remorse', 'sadness', 'surprise', 'neutral', 'label_text'],

In [18]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

def t5_generate(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_length=64,
        do_sample=False
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [19]:
t5_generate("Classify emotion: I am very happy today.")

'happy'

In [21]:
import evaluate
import numpy as np

# Use simple accuracy metric for baseline
accuracy = evaluate.load("accuracy")

def baseline_predict(example):
    prompt = f"Identify the emotion: {example['text']}"
    # Use t5_generate from the previously defined function
    output = t5_generate(prompt)
    return {"prediction": output}

# Get small subset for baseline eval
subset = test_set.select(range(200))
subset = subset.map(baseline_predict)

# Convert label text to single emotion (for simple accuracy)
def clean_prediction(example):
    # Process prediction: take the first emotion if multiple, make it lowercase
    pred_str = example["prediction"].split(",")[0].strip().lower()
    # Process true label: take the first emotion if multiple, make it lowercase
    truth_str = example["label_text"].split(",")[0].strip().lower()

    # Map string labels to integer IDs using the global emotion_labels list
    # Assign -1 if the label is not found in our predefined emotion_labels
    pred_id = emotion_labels.index(pred_str) if pred_str in emotion_labels else -1
    gold_id = emotion_labels.index(truth_str) if truth_str in emotion_labels else -1

    return {
        "pred": pred_id,
        "gold": gold_id
    }

subset = subset.map(clean_prediction)

accuracy.compute(
    predictions=subset["pred"],
    references=subset["gold"]
)


Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

{'accuracy': 0.07}

In [22]:
from transformers import DataCollatorForSeq2Seq

def preprocess(example):
    inputs = ["emotion classification: " + t for t in example["text"]]
    targets = example["label_text"]

    model_inputs = tokenizer(
        inputs,
        max_length=128,
        truncation=True
    )

    labels = tokenizer(
        targets,
        max_length=32,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_train = train_set.map(preprocess, batched=True)
tokenized_val = val_set.map(preprocess, batched=True)

Map:   0%|          | 0/168979 [00:00<?, ? examples/s]

Map:   0%|          | 0/21123 [00:00<?, ? examples/s]

In [25]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

training_args = Seq2SeqTrainingArguments(
    output_dir="./t5-goemotions",
    eval_strategy="epoch", # Changed from evaluation_strategy
    learning_rate=2e-4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    predict_with_generate=True,
    save_strategy="epoch",
)

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

In [39]:
from peft import LoraConfig, get_peft_model
from transformers import BitsAndBytesConfig
import torch

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_2_SEQ_LM"
)

# Define BitsAndBytesConfig for 8-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    bnb_8bit_compute_dtype=torch.float16, # or torch.bfloat16 depending on GPU
    bnb_8bit_quant_type="nf8",
    bnb_8bit_use_double_quant=False,
)

lora_model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    quantization_config=bnb_config, # Pass the quantization config here
    device_map="auto"
)

lora_model = get_peft_model(lora_model, lora_config)
lora_model.print_trainable_parameters()

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


trainable params: 688,128 || all params: 77,649,280 || trainable%: 0.8862


In [ ]:
lora_args = Seq2SeqTrainingArguments(
    output_dir="./lora_t5_goemotions",
    eval_strategy="epoch", # Changed from evaluation_strategy
    learning_rate=2e-4,
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    predict_with_generate=True,
    save_strategy="epoch",
)

lora_trainer = Seq2SeqTrainer(
    model=lora_model,
    args=lora_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    # Removed redundant 'tokenizer' argument as it's passed via data_collator
    data_collator=data_collator,
)

# Train LoRA model
lora_trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


In [ ]:
rouge = evaluate.load("rouge")
bleu = evaluate.load("sacrebleu")
accuracy = evaluate.load("accuracy")

In [ ]:
def evaluate_model(trainer, dataset):
    preds = trainer.predict(dataset)
    pred_text = tokenizer.batch_decode(preds.predictions, skip_special_tokens=True)
    label_text = tokenizer.batch_decode(preds.label_ids, skip_special_tokens=True)

    r = rouge.compute(predictions=pred_text, references=label_text)
    b = bleu.compute(predictions=pred_text, references=[[l] for l in label_text])
    a = accuracy.compute(
        predictions=[p.split(",")[0].lower() for p in pred_text],
        references=[l.split(",")[0].lower() for l in label_text]
    )
    return r, b, a

In [ ]:
lora_results = evaluate_model(lora_trainer, tokenized_val)
lora_results

In [ ]:
import pandas as pd

results_df = pd.DataFrame({
    "Model": ["Baseline", "LoRA"],
    "Accuracy": [baseline_acc['accuracy'], lora_results[2]['accuracy']],
    "ROUGE-L": [0, lora_results[0]['rougeL']],
    "BLEU": [0, lora_results[1]['score']]
})

results_df